In [ ]:
# Step 1: Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectFromModel, RFE
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Step 2: Load dataset
df = pd.read_csv('ds_235795_54.csv')

# Step 3: Data preprocessing
print("Data types of columns:")
print(df.dtypes)

# Step 4: Handle non-numeric columns
X = df.iloc[:, :-1]  # All columns except the last one (features)
y = df.iloc[:, -1]   # Last column (target)

non_numeric_cols = X.select_dtypes(include=['object']).columns
print(f"Non-numeric columns: {non_numeric_cols}")

for col in non_numeric_cols:
    if 'txt' in X[col].values[0]:
        print(f"Dropping column {col} (contains filenames or irrelevant data)")
        X = X.drop(columns=[col])
    else:
        print(f"Applying Label Encoding to column: {col}")
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

if y.dtype == 'object':
    y = LabelEncoder().fit_transform(y)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Feature Selection using RandomForest - SelectFromModel
feature_selector_rf = SelectFromModel(RandomForestClassifier(random_state=42), threshold="mean")
feature_selector_rf.fit(X_train, y_train)
X_train_rf_selected = feature_selector_rf.transform(X_train)
X_test_rf_selected = feature_selector_rf.transform(X_test)
selected_rf_features = X.columns[feature_selector_rf.get_support()]

print(f"Selected features by SelectFromModel (RandomForest): {selected_rf_features}")

# Step 6: Feature Selection using RFE with Gradient Boosting
gb_model = GradientBoostingClassifier(random_state=42)
rfe_selector = RFE(estimator=gb_model, n_features_to_select=10, step=1)
rfe_selector.fit(X_train, y_train)
X_train_rfe_selected = rfe_selector.transform(X_train)
X_test_rfe_selected = rfe_selector.transform(X_test)
selected_rfe_features = X.columns[rfe_selector.get_support()]

print(f"Selected features by RFE (GradientBoosting): {selected_rfe_features}")

# Step 7: Model Training - Random Forest with selected features from SelectFromModel
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_rf_selected, y_train)

# Predictions and evaluation for Random Forest
rf_y_pred = rf_model.predict(X_test_rf_selected)
rf_y_proba = rf_model.predict_proba(X_test_rf_selected)[:, 1]

# Performance evaluation for Random Forest
rf_accuracy = accuracy_score(y_test, rf_y_pred)
rf_roc_auc = roc_auc_score(y_test, rf_y_proba)

print(f"Random Forest Accuracy (Selected Features): {rf_accuracy}")
print(f"Random Forest ROC-AUC (Selected Features): {rf_roc_auc}")
print("Random Forest Classification Report (Selected Features):")
print(classification_report(y_test, rf_y_pred))

# Step 8: Model Training - CatBoost with selected features from RFE
catboost_model = CatBoostClassifier(silent=True, random_state=42)
catboost_model.fit(X_train_rfe_selected, y_train)

# Predictions and evaluation for CatBoost
catboost_y_pred = catboost_model.predict(X_test_rfe_selected)
catboost_y_proba = catboost_model.predict_proba(X_test_rfe_selected)[:, 1]

# Performance evaluation for CatBoost
catboost_accuracy = accuracy_score(y_test, catboost_y_pred)
catboost_roc_auc = roc_auc_score(y_test, catboost_y_proba)

print(f"CatBoost Accuracy (Selected Features): {catboost_accuracy}")
print(f"CatBoost ROC-AUC (Selected Features): {catboost_roc_auc}")
print("CatBoost Classification Report (Selected Features):")
print(classification_report(y_test, catboost_y_pred))

# Step 9: SHAP Analysis for Random Forest
explainer_rf = shap.TreeExplainer(rf_model)
shap_values_rf = explainer_rf.shap_values(X_test_rf_selected)

# Summary plot for Random Forest
plt.figure()
shap.summary_plot(shap_values_rf[1], X_test_rf_selected, feature_names=selected_rf_features)
plt.title("SHAP Summary Plot for Random Forest (Selected Features)")

# Feature importance plot for Random Forest using SHAP values
plt.figure()
shap.summary_plot(shap_values_rf[1], X_test_rf_selected, feature_names=selected_rf_features, plot_type="bar")
plt.title("SHAP Feature Importance (Random Forest)")

# Step 10: SHAP Analysis for CatBoost
explainer_catboost = shap.TreeExplainer(catboost_model)
shap_values_catboost = explainer_catboost.shap_values(X_test_rfe_selected)

# Summary plot for CatBoost
plt.figure()
shap.summary_plot(shap_values_catboost, X_test_rfe_selected, feature_names=selected_rfe_features)
plt.title("SHAP Summary Plot for CatBoost (Selected Features)")

# Feature importance plot for CatBoost using SHAP values
plt.figure()
shap.summary_plot(shap_values_catboost, X_test_rfe_selected, feature_names=selected_rfe_features, plot_type="bar")
plt.title("SHAP Feature Importance (CatBoost)")
